In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [2]:
# 🤖 Carga correcta del dataframe según instrucciones del proyecto
import pandas as pd

df = pd.read_pickle("../02_datos/03_Entrenamiento/01_train_tablon_integrado.pkl")

# 1. Estandarización de nombres de columnas
 
Tras comprobar lo siguiente, se deja igual:
- Conversión a minúsculas
- snake_case
- Eliminación de acentos
- Limpieza de caracteres problemáticos
- Normalización de espacios
- Detección de colisiones tras normalizar
- Resolución de colisiones
- Validación de unicidad final

In [3]:
df.head()

,origen,fuente,no_enviar_email,no_llamar,compra,visitas_total,tiempo_en_site_total,paginas_vistas_visita,ult_actividad,ambito,ocupacion,conociste_google,conociste_revista,conociste_periodico,conociste_youtube,conociste_facebook,conociste_referencias,score_actividad,score_perfil,descarga_lm
id,,,,,,,,,,,,,,,,,,,,
630952,Landing Page Submission,Google,No,No,0,4.000,1221,4.000,Email Opened,Human Resource Management,NaN,No,No,No,No,No,No,NaN,NaN,No
633132,API,Chat,No,No,0,0.000,0,0.000,Chat Conversation,NaN,NaN,No,No,No,No,No,No,NaN,NaN,No
636677,Landing Page Submission,Google,No,No,1,4.000,963,4.000,SMS Sent,Operations Management,Unemployed,No,No,No,No,No,No,14.000,18.000,No
601564,API,Chat,No,No,0,0.000,0,0.000,Email Opened,Select,Student,No,No,No,No,No,No,NaN,NaN,No
645378,Landing Page Submission,Google,No,No,0,3.000,172,3.000,Form Submitted on Website,IT Projects Management,Unemployed,No,No,No,No,No,No,10.000,13.000,No


# 2. Tipos de datos
---
 
Se va a analizar cada columna para:
- Tipo actual y tipo sugerido
- Justificación del tipo sugerido
- % de valores incompatibles
- Pasos previos necesarios para la conversión
 
Se mostrarán los resultados en una tabla resumen.

In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 6365 entries, 630952 to 592736
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   origen                 6365 non-null   str    
 1   fuente                 6340 non-null   str    
 2   no_enviar_email        6365 non-null   str    
 3   no_llamar              6365 non-null   str    
 4   compra                 6365 non-null   int64  
 5   visitas_total          6273 non-null   float64
 6   tiempo_en_site_total   6365 non-null   int64  
 7   paginas_vistas_visita  6273 non-null   float64
 8   ult_actividad          6297 non-null   str    
 9   ambito                 5353 non-null   str    
 10  ocupacion              4477 non-null   str    
 11  conociste_google       6365 non-null   str    
 12  conociste_revista      6365 non-null   str    
 13  conociste_periodico    6365 non-null   str    
 14  conociste_youtube      6365 non-null   str    
 15  conociste_fac

In [5]:
# Modificamos tipos de variables

df['visitas_total'] = df['visitas_total'].astype('Int64')

# 3. Valores únicos
---

### 3.1. Valores únicos

In [6]:
# Detectamos valores únicos y revisamos cardinalidad
df.nunique().sort_values()

conociste_youtube           1
conociste_periodico         1
conociste_revista           1
descarga_lm                 2
conociste_referencias       2
no_enviar_email             2
no_llamar                   2
compra                      2
conociste_facebook          2
conociste_google            2
origen                      4
ocupacion                   6
score_perfil               10
score_actividad            12
ult_actividad              15
fuente                     15
ambito                     19
visitas_total              35
paginas_vistas_visita      95
tiempo_en_site_total     1572
dtype: int64

Detectamos que tanto `conociste_youtube` como `conociste_periodico` son variables, ya que contienen solo adquiere 1 tipo de dato. Se eliminarán, ya que no aporta información.

Por otro lado, observamos que `ambito` es una variable que tiene bastante posibilidades, por lo que, tiene alta cardinalidad que ya las procesaremos más adelante viendo la distribución de las posibilidades.

In [7]:
df.drop(columns=['conociste_youtube', 'conociste_revista'], inplace=True)

### 3.2. Duplicados

In [9]:
# Detectamos duplicados
df.duplicated().sum()

np.int64(0)

In [10]:
# Eliminamos duplicados
df.drop_duplicates(inplace=True)

# 4. Transformación de variables
---

In [11]:
# Creamos datases de variables categóricas y numéricas

cat = df.select_dtypes(exclude='number').copy()

num = df.select_dtypes(include='number').copy()

## 4.1. Categóricas

Vamos a proceder con el análisis y gestión de las variables categóricas. Se observa que existen 2 variables que tienen varios nulos.

In [12]:
# Identificacion de nulos en categóricas

cat.isna().sum().sort_values(ascending=False)

ocupacion                1360
ambito                    484
ult_actividad              68
fuente                     17
origen                      0
no_enviar_email             0
no_llamar                   0
conociste_google            0
conociste_periodico         0
conociste_facebook          0
conociste_referencias       0
descarga_lm                 0
dtype: int64

Detectamos que `ocupacion` y `ambito` poseen bastantes nulos pero ambas variables son muy relevantes, por lo que, no eliminaremos ninguna variable ni ninguna variable. 

Se podrían eliminar los registros marcando un thresh=8 (al tener 4 variables) pero eso eliminaría todos los registros en el dataset de `num` ya que contiene solo 4 columnas, por lo que, se procederá **imputando** y para cada uno se estudiará el caso.

#### Ocupación
Tras analizar sus valores, vemos que la moda será `Unemployed` por lo que, parece una buena variable a **imputar por la moda**

In [13]:
cat.ocupacion.value_counts()

ocupacion
Unemployed              3356
Working Professional     451
Student                  128
Other                      9
Housewife                  7
Businessman                6
Name: count, dtype: int64

#### Ámbito
Tras analizar sus valores, vemos que la moda será `ambito` por lo que, parece una buena variable a imputar y lo metemos dentro de `Select` que es equivalente a no tener información

In [14]:
cat.ambito.value_counts()

ambito
Select                               840
Finance Management                   660
Marketing Management                 563
Human Resource Management            558
Operations Management                352
Business Administration              256
IT Projects Management               255
Supply Chain Management              245
Banking, Investment And Insurance    236
Media and Advertising                151
Travel and Tourism                   134
International Business               124
Healthcare Management                112
Hospitality Management                83
E-COMMERCE                            77
Retail Management                     68
Rural and Agribusiness                48
E-Business                            42
Services Excellence                   29
Name: count, dtype: int64

#### Última actividad
Tras analizar sus valores, se observa que en la variable `ult_actividad` no tiene sentido **imputar por la moda** porque imputar algo que no sabemos a una categoría tan relevante. Se puede imputar como "desconocido".

In [15]:
cat.ult_actividad.value_counts()

ult_actividad
Email Opened                    2035
SMS Sent                        1631
Page Visited on Website          435
Converted to Lead                289
Chat Conversation                284
Email Bounced                    199
Email Link Clicked               151
Form Submitted on Website         85
Unreachable                       65
Unsubscribed                      38
Had a Phone Conversation          25
Approached upfront                 6
View in browser link Clicked       3
Email Marked Spam                  2
Visited Booth in Tradeshow         1
Name: count, dtype: int64

#### Fuente
Tras analizar sus valores, se observa que en la variable `fuente` **imputar por la moda** no estaría tan mal, ya que en la mayoría de los casos viene por Google pero en este caso no se hará porque los contetos de las 2 primeras opciones, son bastante similares y no tenemos criterio meter en `tráfico directo`. Por lo que, pondremos una categoría de "desconocida"

In [16]:
cat.fuente.value_counts()

fuente
Google               2012
Direct Traffic       1787
Organic Search        805
Chat                  324
Reference             233
Referral Sites         88
Facebook               34
bing                    4
google                  4
Live Chat               2
Press_Release           2
Click2call              2
Pay per Click Ads       1
Social Media            1
youtubechannel          1
Name: count, dtype: int64

Vamos a incluir todas las variables, ya que en caso de que los nuevos datos sean nulos, cuando esté en producción, se imputará por el valor "DESCONOCIDO" y así podemos gestionar estos nulos, ya que sino `scikit-learn` se romperá.

In [35]:
# Imputar valor

var_imputar_valor = ['origen','fuente','no_enviar_email','no_llamar','ult_actividad','conociste_google','conociste_periodico','conociste_facebook', 'conociste_referencias','descarga_lm']

valor = 'DESCONOCIDO'

In [37]:
cat[var_imputar_valor] = cat[var_imputar_valor].fillna(valor)

In [38]:
# Imputar por la moda
variables_imputar_moda = ['ocupacion', 'ambito']
cat[variables_imputar_moda]

def imputar_moda(variables):
    for variable in variables:
        cat[variable] = cat[variable].fillna(cat[variable].mode()[0])

imputar_moda(variables_imputar_moda)

Tras realizar la imputacion, ya no tenemos ningún nulo en las variables.

In [39]:
cat.isna().sum().sort_values()

origen                   0
fuente                   0
no_enviar_email          0
no_llamar                0
ult_actividad            0
ambito                   0
ocupacion                0
conociste_google         0
conociste_periodico      0
conociste_facebook       0
conociste_referencias    0
descarga_lm              0
dtype: int64

### 4.1.1. Atípicos en variables categóricas

In [40]:
def agrupar_cat_raras(variable, criterio = 0.05):
    #Calcula las frecuencias
    frecuencias = variable.value_counts(normalize=True)

    #Identifica las que están por debajo 
    raras = frecuencias[frecuencias < criterio].index

    return np.where(variable.isin(raras), 'OTROS', variable)


Tras definir la función, agrupamos las categorías que están por debajo del 2% de representación:

In [41]:
variables_agrupar_categorias = cat.columns.to_list()
criterio_agrupar = 0.02

for variable in variables_agrupar_categorias:
    cat[variable] = agrupar_cat_raras(cat[variable], criterio=criterio_agrupar)

## 4.2. Numéricas

In [102]:
num.isna().sum().sort_values(ascending=False)

score_actividad          2366
score_perfil             2366
visitas_total              92
paginas_vistas_visita      92
compra                      0
tiempo_en_site_total        0
dtype: int64

Detectamos que `score_actividad` y `score_perfil` son variables que tienen bastantes nulos pero no sabemos si puede ser predictiva, quizás si puede dar información.

Respecto a las variables de `visitas_total` y `paginas_visita_visita` vemos que tiene nulos también. Lo que se hará con estas variables es imputar por la moda, al igual que se hacía antes ya que es mas garantía de éxito frente a distribuciones sesgadas. Incluso se van a imputar las variables `compra` y `tiempo_en_site_total` porque aunque aquí vengan sin nulos, no indica que cuando esté en producción, venga sin nulos y **podría romperse el sistema**

In [87]:
var_num_imputar_moda = num.columns.to_list()

In [90]:
def imputar_mediana (variable):
    if pd.api.types.is_int64_dtype:
        return(variable.fillna(int(variable.median())))
    
    else:
        return variable.fillna(variable.median())

num[var_num_imputar_moda] = num[var_num_imputar_moda].apply(imputar_mediana)

In [91]:
num.isna().sum().sort_values(ascending=False)

compra                   0
visitas_total            0
tiempo_en_site_total     0
paginas_vistas_visita    0
score_actividad          0
score_perfil             0
dtype: int64

### 4.2.1. Atípicos en variables numéricas
Para la detección de atípicos en variables numéricas, definiremos 4 desviaciones típicos para su detección

In [92]:
num_desv_tip = 4

In [93]:
# Función que establece los atípicos por límites

def atipicos_des_tip(variable, num_desv_tip = 4):
    # Sacamos los nulos por ahora
    variable = variable.dropna()

    # Calculamos los límites
    media = np.mean(variable)
    sd = np.std(variable)
    umbral = sd * num_desv_tip

    lim_inf = media - umbral
    lim_sup = media + umbral

    # Encontramos los indices de los que están fuera de los límites
    return variable[(variable < lim_inf) | (variable > lim_sup)].index.tolist()

In [94]:
# Función que cuenta el número de atípicos por variable

def conteo_atipicos(df, variable, num_desv_tip=4):
    atipicos = atipicos_des_tip(df[variable], num_desv_tip)
    return (df.loc[atipicos, variable].value_counts())

Ahora vamos a aplicar estas variables a los atípicos que son las variables que antes habíamos contado: `edad`, `num_contatcos_esta_campana`

In [104]:
var_atipicos_dt = num.columns.to_list()

In [105]:
for variable in var_atipicos_dt:
    print('\n' + variable + ':\n')
    print(conteo_atipicos(num, variable, num_desv_tip))


compra:

Series([], Name: count, dtype: int64)

visitas_total:

visitas_total
25     4
27     2
26     2
29     2
30     1
28     1
251    1
141    1
41     1
54     1
Name: count, dtype: Int64

tiempo_en_site_total:

Series([], Name: count, dtype: int64)

paginas_vistas_visita:

paginas_vistas_visita
11.000    8
14.000    6
13.000    5
12.000    5
16.000    2
15.000    1
14.500    1
24.000    1
11.500    1
Name: count, dtype: int64

score_actividad:

score_actividad
10.000    43
18.000     4
9.000      3
8.000      2
7.000      1
Name: count, dtype: int64

score_perfil:

Series([], Name: count, dtype: int64)


**Conclusiones**
* `visitas_total`: ponemos máx 50
* `paginas_vistas_visita`: ponemos máx 20
* `score_actividad`: no se modifica, ya que entendemos que se calcula por el software de CRM y que lo da bien, por lo que, los atípicos tienen sentido.

Pero esto lo vamos a corregir desde un enfoque de **Negocio**, por lo que, lo corregiremos con *Winsorización*

In [ ]:
# Aplicamos Winsorización manual

df['visitas_total'] = df['visitas_total'].clip(0, 50)
df['paginas_vistas_visita'] = df['paginas_vistas_visita'].clip(0, 20)

Se va a crear una nueva columna de `usuario_nuevo`, ya que, cuando esté en producción, tiene sentido identificar si no tiene valor, lo que significaría que es **usuario nuevo**

In [ ]:
# Imputar nulos en score_actividad y score_perfil por ceros
cols_imputar = ['score_actividad', 'score_perfil']

imputado = df[cols_imputar].isnull().any(axis=1).astype(int)

# crear variable usuario_nuevo
df['usuario_nuevo'] = imputado

## 5. Exportamos dataset limpios
---

In [120]:
# Definimos rutas

nombre_df = "02_train_tablon_calidad.pkl"
ruta_proyecto = "../02_datos/03_Entrenamiento/"

In [121]:
# Guardar el dataframe limpio para la siguiente fase
ruta_trabajo = ruta_proyecto + nombre_df
df.to_pickle(ruta_trabajo)
print(f"✅ Dataframe {nombre_df} guardado en {ruta_trabajo}")

✅ Dataframe 02_train_tablon_calidad.pkl guardado en ../02_datos/03_Entrenamiento/02_train_tablon_calidad.pkl


In [122]:
# Guardar datasets limpios para la siguiente fase
ruta_cat = ruta_proyecto + "02b_cat_resultado_calidad.pkl"
ruta_num = ruta_proyecto + "02c_num_resultado_calidad.pkl"

cat.to_pickle(ruta_cat)
num.to_pickle(ruta_num)

print(f"✅ Dataframe cat guardado en {ruta_cat}")
print(f"✅ Dataframe num guardado en {ruta_num}")

✅ Dataframe cat guardado en ../02_datos/03_Entrenamiento/02b_cat_resultado_calidad.pkl
✅ Dataframe num guardado en ../02_datos/03_Entrenamiento/02c_num_resultado_calidad.pkl


In [123]:
# Mostrar estructura final del dataframe para documentación
df.info()

<class 'pandas.DataFrame'>
Index: 5317 entries, 630952 to 592736
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   origen                 5317 non-null   str    
 1   fuente                 5300 non-null   str    
 2   no_enviar_email        5317 non-null   str    
 3   no_llamar              5317 non-null   str    
 4   compra                 5317 non-null   int64  
 5   visitas_total          5225 non-null   Int64  
 6   tiempo_en_site_total   5317 non-null   int64  
 7   paginas_vistas_visita  5225 non-null   float64
 8   ult_actividad          5249 non-null   str    
 9   ambito                 4833 non-null   str    
 10  ocupacion              3957 non-null   str    
 11  conociste_google       5317 non-null   str    
 12  conociste_periodico    5317 non-null   str    
 13  conociste_facebook     5317 non-null   str    
 14  conociste_referencias  5317 non-null   str    
 15  score_activid